# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

*Entities in the dataset are referenced by their `@id` fields.*

In [ ]:
# List all record sets and associated field IDs
print("Available record sets and fields (by @id):\n")
record_sets = []
for record_set in metadata.record_sets:
    record_sets.append(record_set.id_)
    print(f"- RecordSet @id: {record_set.id_}")
    print(f"  Name: {record_set.name}")
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"    - Field @id: {field.id_} (name: {field.name}, dataType: {getattr(field, 'data_type', 'N/A')})")
    print()

# Display one or more sample records for each record set (@id form)
for record_set_id in record_sets:
    print(f"Sample record from RecordSet @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint(record)
        if i == 0:
            break  # Just show the first example
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns for RecordSet: {record_set_id}")

# For demonstration, pick the first available record set
main_record_set = record_sets[0] if record_sets else None
if main_record_set:
    print(f"\nColumns in DataFrame ({main_record_set}):\n", dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All analysis below references columns by their Croissant `@id`.

In [ ]:
import numpy as np

# You can update these IDs based on the record set field overview
# For this dataset, let's assume 'accession', 'description', 'psm_count' are some possible @id column names
# We'll programmatically select a numeric field
df = dataframes[main_record_set]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

print(f"Using numeric field @id for EDA: {numeric_field_id}\n")

# Filtering
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if len(df) and not np.isnan(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} above mean (>{threshold:.2f}):\n{filtered_df.head()}\n")

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(), "\n")

    # Grouping (pick a non-numeric group field if available)
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field
if numeric_field_id and len(df[numeric_field_id].dropna()) > 1:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# If there is a group field, visualize boxplot
if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use `mlcroissant` to load and explore the FAIR² dataset described by a Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). You learned how to reference dataset entities by their `@id`, examine record set structure, extract tabular data, and run basic EDA and data visualizations. For deeper investigation, check dataset documentation or further analyze specific fields and relationships using the same @id-centric workflow!